# Node Classification

## Task Description
Classify nodes in a graph using a Graph Convolutional Network (GCN).

In this notebook, we will:
1. Install libraries.
2. Import modules.
3. Load Cora dataset.
4. Data Cleaning.
5. Load GCN Model.
6. Inference (Train/Test).
7. Save Results.
8. Advanced Usage.


## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## 1. Installation
Install `torch_geometric` and dependencies.

In [ ]:
!pip install -q torch_geometric

## 2. Imports
Import PyG modules.

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.datasets import Planetoid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 3. Load Data
Load the Cora dataset.

In [ ]:
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0].to(device)
print(f"Dataset: {dataset}: {len(dataset)} graphs")
print(f"Data keys: {data.keys}")

## 4. Data Cleaning
Standard normalization is handled by the dataset loader, but we inspect features.

In [ ]:
print(f"Num features: {dataset.num_node_features}")
print(f"Num classes: {dataset.num_classes}")

## 5. Load Model
Define a simple GCN.

In [ ]:
class GCN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(dataset.num_node_features, 16)
        self.conv2 = GCNConv(16, dataset.num_classes)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

model = GCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print(model)

## 6. Inference
Train the model and evaluate.

In [ ]:
model.train()
for epoch in range(200):
    optimizer.zero_grad()
    out = model(data)
    loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()

model.eval()
pred = model(data).argmax(dim=1)
correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
acc = int(correct) / int(data.test_mask.sum())
print(f'Accuracy: {acc:.4f}')

## 7. Save Results
Save accuracy.

In [ ]:
results = {'accuracy': acc}
with open('results.txt', 'w') as f:
    f.write(str(results))
print('Results saved to results.txt')

# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
     rate = save_rate if 'save_rate' in locals() else 16000
     try:
         wavfile.write('generated_audio.wav', rate, audio_data)
         print("Saved generated_audio.wav")
     except:
         pass


## 8. Advanced Usage
Try `GATConv` (Graph Attention Networks) for better performance.

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

In [ ]:
print("Try replacing GCNConv with GATConv!")